# 4. Scaled Dot-Product Attention с нуля

**Цель:** Реализовать механизм внимания с нуля на PyTorch, понять Query/Key/Value, визуализировать матрицу внимания и сравнить с оптимизированной реализацией.

---

In [ ]:
import sys, os, logging, math
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("attention")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
log.info("Using device: %s", device)

## 4.1 Постановка задачи: зачем нужно внимание?

**Проблема RNN/LSTM:**
- Скрытое состояние сжимает всю информацию о последовательности в один вектор
- При длинных последовательностях информация "забывается" (vanishing gradient)
- Нет прямого доступа к произвольным позициям входа

**Идея внимания (Bahdanau, 2014):**
- На каждом шаге декодирования "смотрим" на все позиции входа
- Решаем, какие части входа наиболее релевантны для текущего шага
- Взвешенная сумма скрытых состояний энкодера

**Scaled Dot-Product Attention (Vaswani et al., 2017):**
- Query (запрос) — что мы ищем
- Key (ключ) — что у нас есть
- Value (значение) — информация, которую мы хотим извлечь

## 4.2 Query, Key, Value — интуиция и математика

**Аналогия (поиск в словаре):**
- **Query** — слово, которое мы ищем
- **Key** — заголовки словарных статей
- **Value** — содержимое статей
- **Score** (Q·Kᵀ) — насколько хорошо запрос соответствует каждому ключу
- **Softmax** — нормализация scores в распределение вероятностей
- **Output** = взвешенная сумма Values по этим весам

**Математика:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Зачем делить на $\sqrt{d_k}$? Чтобы дисперсия scores оставалась ~1, и softmax не уходил в режим "почти one-hot" или "почти uniform" при большой размерности.

In [ ]:
log.debug("Implementing scaled dot-product attention from scratch")

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention.
    
    Args:
        Q: (batch, seq_len_q, d_k)
        K: (batch, seq_len_k, d_k)
        V: (batch, seq_len_k, d_v)
        mask: (batch, seq_len_q, seq_len_k) или (seq_len_q, seq_len_k) — True там, где нужно замаскировать
    
    Returns:
        output: (batch, seq_len_q, d_v)
        attention_weights: (batch, seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)
    
    # 1. Считаем scores: Q @ K^T
    scores = Q @ K.transpose(-2, -1)  # (batch, seq_len_q, seq_len_k)
    
    # 2. Масштабирование
    scores = scores / math.sqrt(d_k)
    log.debug("Scores shape: %s, d_k=%d", scores.shape, d_k)
    
    # 3. Маскировка (если есть)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
        log.debug("Mask applied, masked positions: %d", mask.eq(0).sum().item())
    
    # 4. Softmax по ключам
    attention_weights = F.softmax(scores, dim=-1)
    log.debug("Attention weights shape: %s", attention_weights.shape)
    
    # 5. Взвешенная сумма Values
    output = attention_weights @ V  # (batch, seq_len_q, d_v)
    log.debug("Output shape: %s", output.shape)
    
    return output, attention_weights

In [ ]:
log.debug("Testing attention on simple random data")

batch_size, seq_len, d_k, d_v = 2, 4, 8, 8
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_v)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape:     {Q.shape}")
print(f"K shape:     {K.shape}")
print(f"V shape:     {V.shape}")
print(f"Output shape:{output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nAttention weights (batch 0):\n{attn_weights[0].detach().numpy().round(3)}")
print(f"\nRow sums (should be ~1.0): {attn_weights[0].sum(dim=-1).numpy().round(4)}")

log.info("Basic attention test passed")

## 4.3 Визуализация матрицы внимания

Heatmap — основной инструмент анализа. Показывает, какие токены "смотрят" на какие.

In [ ]:
log.debug("Visualizing attention heatmap")

# Создаём осмысленные данные: ищем соответствия
seq_len = 6
d_k = 4

# Q — запросы для каждой позиции
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

# Делаем так, чтобы первые 3 позиции "смотрели" на последние 3
Q[:, :3, :] = 0
K[:, 3:, :] = 0
Q[:, :3, :] = Q[:, 3:, :] * 2  # увеличиваем сходство

output, attn_weights = scaled_dot_product_attention(Q, K, V)

plt.figure(figsize=(8, 6))
plt.imshow(attn_weights[0].detach().numpy(), cmap='Blues')
plt.colorbar(label='Attention weight')
plt.xlabel('Key positions')
plt.ylabel('Query positions')
plt.title('Attention Heatmap')
plt.xticks(range(seq_len))
plt.yticks(range(seq_len))
for i in range(seq_len):
    for j in range(seq_len):
        val = attn_weights[0, i, j].item()
        plt.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9)
plt.tight_layout()
plt.show()
log.info("Attention heatmap plotted")

## 4.4 Маскировка: Padding Mask

В реальных данных последовательности имеют разную длину. Мы паддим (дополняем) короткие последовательности специальным PAD-токеном и маскируем его, чтобы он не влиял на attention.

In [ ]:
log.debug("Demonstrating padding mask")

batch, seq_len, d_k = 2, 5, 8
# Две последовательности разной длины: длины 3 и 5
lengths = torch.tensor([3, 5])

# Создаём маску: True для валидных позиций
mask = torch.arange(seq_len).unsqueeze(0) < lengths.unsqueeze(1)  # (batch, seq_len)
mask = mask.unsqueeze(1)  # (batch, 1, seq_len) — для broadcasting с scores
print(f"Padding mask:\n{mask}")

Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i in range(2):
    ax = axes[i]
    im = ax.imshow(attn_weights[i].detach().numpy(), cmap='Blues')
    ax.set_title(f'Sequence {i+1} (length {lengths[i].item()})')
    ax.set_xlabel('Key positions')
    ax.set_ylabel('Query positions')
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()
log.info("Padding mask visualization complete")

## 4.5 Демонстрация на синтетических данных: поиск соответствий

Создадим задачу, где внимание естественно возникает: нужно найти, какие элементы одной последовательности соответствуют элементам другой.

In [ ]:
log.debug("Running attention demo on synthetic matching task")

# Генерируем последовательности с паттернами
torch.manual_seed(42)
n_patterns = 4
d_k = 16
seq_len = 6

# Шаблоны: base + noise
patterns = torch.randn(n_patterns, d_k)

# Создаём Q и K, где Q[i] похож на один из patterns, а K[j] — на другой
Q = patterns[torch.randint(0, n_patterns, (seq_len,))] + 0.1 * torch.randn(seq_len, d_k)
K = patterns[torch.randint(0, n_patterns, (seq_len,))] + 0.1 * torch.randn(seq_len, d_k)
V = K.clone()  # Value = Key (Identity attention)

Q, K, V = Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

plt.figure(figsize=(8, 6))
plt.imshow(attn_weights[0].detach().numpy(), cmap='Blues')
plt.colorbar(label='Attention weight')
plt.title('Attention: Pattern Matching')
plt.xlabel('Key position')
plt.ylabel('Query position')
for i in range(seq_len):
    for j in range(seq_len):
        val = attn_weights[0, i, j].item()
        plt.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.show()
log.info("Pattern matching attention demo complete")

## 4.6 Сравнение с оптимизированной реализацией PyTorch

PyTorch предоставляет `torch.nn.functional.scaled_dot_product_attention`. Сравним результаты.

In [ ]:
log.debug("Comparing manual implementation with torch's optimized version")

torch.manual_seed(123)
batch, seq_len, d_k = 4, 8, 32
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

# Наша реализация
output_manual, attn_manual = scaled_dot_product_attention(Q, K, V)

# PyTorch реализация
output_torch = F.scaled_dot_product_attention(Q, K, V)

# Сравнение
diff = (output_manual - output_torch).abs().max().item()
print(f"Max difference: {diff:.2e}")
print(f"Outputs match: {torch.allclose(output_manual, output_torch, atol=1e-6)}")
log.info("Comparison with F.scaled_dot_product_attention: max_diff=%.2e", diff)

In [ ]:
# Бенчмарк скорости
import time

log.debug("Running performance benchmark")

batch, seq_len, d_k = 16, 128, 64
Q = torch.randn(batch, seq_len, d_k, device=device)
K = torch.randn(batch, seq_len, d_k, device=device)
V = torch.randn(batch, seq_len, d_k, device=device)

def bench(fn, name, n_runs=20):
    # warmup
    for _ in range(3):
        fn()
    if device.type == 'mps':
        torch.mps.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_runs):
        fn()
    if device.type == 'mps':
        torch.mps.synchronize()
    elapsed = (time.perf_counter() - start) / n_runs
    print(f"{name:30s}: {elapsed*1000:.3f} ms")
    log.debug("Benchmark %s: %.3f ms", name, elapsed*1000)
    return elapsed

manual_time = bench(lambda: scaled_dot_product_attention(Q, K, V), "Manual attention")
torch_time = bench(lambda: F.scaled_dot_product_attention(Q, K, V), "F.scaled_dot_product_attention")
print(f"Speedup: {manual_time / torch_time:.1f}x")

## 4.7 Визуализация: влияние масштабирования на softmax

Демонстрируем, почему деление на $\sqrt{d_k}$ критично.

In [ ]:
log.debug("Visualizing effect of scaling on softmax distribution")

dims = [1, 8, 64, 256]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, d_k in zip(axes.flat, dims):
    Q = torch.randn(1, 10, d_k)
    K = torch.randn(1, 10, d_k)
    
    scores = Q @ K.transpose(-2, -1)
    scores_scaled = scores / math.sqrt(d_k)
    
    probs = F.softmax(scores, dim=-1)[0, 0].detach().numpy()
    probs_scaled = F.softmax(scores_scaled, dim=-1)[0, 0].detach().numpy()
    
    ax.hist(probs, bins=20, alpha=0.5, label='Without scaling')
    ax.hist(probs_scaled, bins=20, alpha=0.5, label='With scaling')
    ax.set_title(f'd_k = {d_k}')
    ax.set_xlabel('Attention weight')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    log.debug("d_k=%d: entropy unscaled=%.3f, scaled=%.3f",
              d_k,
              -((probs+1e-10)*np.log(probs+1e-10)).sum(),
              -((probs_scaled+1e-10)*np.log(probs_scaled+1e-10)).sum())

plt.suptitle('Effect of Scaling on Softmax Distribution', fontsize=14)
plt.tight_layout()
plt.show()
log.info("Scaling effect visualization complete")

In [ ]:
print("=== Scaled Dot-Product Attention complete ===")
print("Topics covered:")
print("  - Scaled Dot-Product Attention from scratch")
print("  - Q, K, V intuition and mathematics")
print("  - Attention heatmap visualization")
print("  - Padding mask for variable-length sequences")
print("  - Pattern matching demo on synthetic data")
print("  - Comparison with F.scaled_dot_product_attention")
print("  - Performance benchmark")
print("  - Why scaling by sqrt(d_k) matters")
log.info("Attention notebook complete")